# Clean Full Enriched: 3 Algorithms (Full Dataset)

Ноутбук показывает пример запуска 3 алгоритмов из `src/flowopt` на полном датасете `clean_full_enriched`:
- `real_gap_vrp`
- `real_milp`
- `real_genetic`

Внимание: полный прогон может работать долго.


In [ ]:
from __future__ import annotations

import json
import time
from datetime import datetime
from pathlib import Path

import pandas as pd

from flowopt.pipeline import (
    DATA_ROOT,
    run_real_gap_vrp,
    run_real_milp,
    run_real_genetic,
)

REPO_ROOT = Path.cwd().resolve()
REAL_ROOT = DATA_ROOT / "real" / "clean_full_enriched"
DATASET_PATH = REAL_ROOT / "dataset_real_spb_clean_full_enriched.json"
README_PATH = REAL_ROOT / "README.md"
METRICS_SUMMARY_PATH = REAL_ROOT / "dataset_real_spb_clean_full_enriched_metrics_summary.json"

print("REPO_ROOT:", REPO_ROOT)
print("DATASET_PATH:", DATASET_PATH)
print("dataset exists:", DATASET_PATH.exists())
print("README exists:", README_PATH.exists())
print("metrics summary exists:", METRICS_SUMMARY_PATH.exists())


In [ ]:
if METRICS_SUMMARY_PATH.exists():
    metrics_summary = json.loads(METRICS_SUMMARY_PATH.read_text(encoding="utf-8"))
    print("dataset_profile:", metrics_summary.get("dataset_profile"))
    print("counts:", metrics_summary.get("counts", {}))
    print("fleet:", {
        "payload_total_tons": metrics_summary.get("fleet", {}).get("payload_total_tons"),
        "raw_volume_total_m3": metrics_summary.get("fleet", {}).get("raw_volume_total_m3"),
    })
    print("objects:", {
        "day_capacity_tons_total": metrics_summary.get("objects", {}).get("day_capacity_tons_total"),
        "day_capacity_volume_m3_total": metrics_summary.get("objects", {}).get("day_capacity_volume_m3_total"),
    })


In [ ]:
RUN_GAP = True
RUN_MILP = True
RUN_GENETIC = True

# GAP-VRP params
GAP_STEP1_METHOD = "greedy"   # "lp" тоже можно, но обычно дольше
GAP_ITER = 20
GAP_MAX_AGENTS = 160           # ограничение для ускорения full-прогона

# MILP params
MILP_TIME_LIMIT_SEC = 120
MILP_UNASSIGNED_PENALTY = 1e5

# Genetic params
GA_POPULATION_SIZE = 20
GA_GENERATIONS = 24
GA_GENERATION_SCALE = 0.5
GA_ELITE_SIZE = 4
GA_SEED = 42

SHOW_PROGRESS = True


In [ ]:
def _run_with_live_status(name, fn, **kwargs):
    t0 = time.perf_counter()

    def progress(msg: str):
        dt = time.perf_counter() - t0
        print(f"[+{dt:7.1f}s] [{name}] {msg}", flush=True)

    print(f"\n=== {name}: START ===", flush=True)
    metrics = fn(progress_hook=progress, show_progress=SHOW_PROGRESS, **kwargs)
    dt = time.perf_counter() - t0
    print(f"=== {name}: DONE in {dt:.1f}s ===", flush=True)
    return metrics

results = []
if RUN_GAP:
    results.append(
        _run_with_live_status(
            "real_gap_vrp",
            run_real_gap_vrp,
            dataset_path=DATASET_PATH,
            step1_method=GAP_STEP1_METHOD,
            gap_iter=GAP_ITER,
            max_agents=GAP_MAX_AGENTS,
            use_repair=True,
            verbose=False,
        )
    )

if RUN_MILP:
    results.append(
        _run_with_live_status(
            "real_milp",
            run_real_milp,
            dataset_path=DATASET_PATH,
            time_limit_sec=MILP_TIME_LIMIT_SEC,
            unassigned_penalty=MILP_UNASSIGNED_PENALTY,
        )
    )

if RUN_GENETIC:
    results.append(
        _run_with_live_status(
            "real_genetic",
            run_real_genetic,
            dataset_path=DATASET_PATH,
            population_size=GA_POPULATION_SIZE,
            generations=GA_GENERATIONS,
            generation_scale=GA_GENERATION_SCALE,
            elite_size=GA_ELITE_SIZE,
            seed=GA_SEED,
        )
    )


In [ ]:
if not results:
    raise RuntimeError("No algorithms selected")

summary = pd.DataFrame([m.as_dict() for m in results])
cols = [
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "runtime_sec",
    "payload_utilization_pct_avg",
    "volume_utilization_pct_avg",
    "object_mass_overload_count",
    "object_volume_overload_count",
    "deadhead_km",
    "deadhead_share_pct",
    "total_km",
    "total_hours",
    "solver_error",
]
for c in cols:
    if c not in summary.columns:
        summary[c] = None
summary[cols]


In [ ]:
out_dir = REPO_ROOT / "notebooks" / "local" / "clean_full_enriched"
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_json = out_dir / f"full_3algo_{stamp}.json"
out_csv = out_dir / f"full_3algo_{stamp}.csv"

payload = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset_path": str(DATASET_PATH),
    "params": {
        "gap": {"step1_method": GAP_STEP1_METHOD, "gap_iter": GAP_ITER, "max_agents": GAP_MAX_AGENTS},
        "milp": {"time_limit_sec": MILP_TIME_LIMIT_SEC, "unassigned_penalty": MILP_UNASSIGNED_PENALTY},
        "genetic": {
            "population_size": GA_POPULATION_SIZE,
            "generations": GA_GENERATIONS,
            "generation_scale": GA_GENERATION_SCALE,
            "elite_size": GA_ELITE_SIZE,
            "seed": GA_SEED,
        },
    },
    "results": [m.as_dict() for m in results],
}
out_json.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
summary.to_csv(out_csv, index=False)
print("saved:", out_json)
print("saved:", out_csv)
